# Week 4: Vector Power-Up & Raster Integration

**Student Worksheet** — Fill in the code cells using AI assistance or your own code.

This week you will:
1. Master vector aggregation (dissolve & groupby)
2. Understand raster data as NumPy arrays with spatial metadata
3. Compute terrain slope from DEM
4. Extract raster values into vector shapes (zonal statistics)

**Packages needed:** `geopandas`, `rioxarray`, `rasterstats`, `numpy`, `matplotlib`

> If you haven't installed them yet, run:  
> `pip install rioxarray rasterstats geopandas folium mapclassify matplotlib`

## Cell [1] — Environment Setup

Import all necessary packages and load Week 3 township data.

In [7]:
# Cell [1]: YOUR CODE HERE

# Import: geopandas, pandas, numpy, matplotlib.pyplot, rioxarray, xarray
# Load TGOS townships (remember: layer='TOWN_MOI_1140318')
# Convert to EPSG:3826
# Print shape, CRS, and first 5 rows
import geopandas as gpd
import matplotlib.pyplot as plt
import rioxarray
import xarray as xr
import numpy as np
import pandas as pd
from urllib.parse import quote

TGOS_BASE = 'https://www.tgos.tw/tgos/VirtualDir/Product/3fe61d4a-ca23-4f45-8aca-4a536f40f290/'
FILENAME  = '鄉(鎮、市、區)界線1140318.zip'
url = TGOS_BASE + quote(FILENAME)

print('Reading township boundaries from TGOS ...')
townships = gpd.read_file(url, layer='TOWN_MOI_1140318')
print('Done!')

# Convert to EPSG:3826
townships = townships.to_crs('EPSG:3826')

# Print shape and CRS
print(f'Shape: {townships.shape}')
print(f'CRS: {townships.crs}')
print(townships.head())

Reading township boundaries from TGOS ...


HTTPError: HTTP Error 403: Forbidden

## Cell [2] — Dissolve & Groupby

**Dissolve**: Merge 368 townships → ~22 counties  
**Groupby**: Count shelters per county and sum capacity

In [ ]:
# Cell [2]: YOUR CODE HERE
# 1. Dissolve townships by 'COUNTYNAME' → counties GeoDataFrame
# 2. Print: how many counties? What geometry types?
# 3. Calculate area in km² for each county
# 4. Show the top 5 largest counties

# Dissolve townships by county name
counties = townships.dissolve(by='COUNTYNAME')

print(f"Number of counties: {len(counties)}")
print(f"Geometry types: {counties.geometry.type.unique()}")

# Calculate area in km² (EPSG:3826 uses meters as units)
counties['area_km2'] = counties.geometry.area / 1_000_000

# Show top 5 largest counties
print("\nTop 5 largest counties by area:")
top5_counties = counties.nlargest(5, 'area_km2')[['area_km2']]
print(top5_counties)

---

## 🔬 Lab 1: Vector Aggregation (20 minutes)

**Goal**: Practice dissolve & groupby before entering the raster world.

> **⚠️ 注意**：Lab 1 使用**合成資料**（synthetic data）練習 dissolve/groupby 操作。課後作業（ARIA v2.0）會使用**真實的避難收容所 CSV**，操作方式相同但資料來源不同。

### Step 1: Dissolve townships → counties

In [ ]:
# Lab 1 Step 1: YOUR CODE HERE
# Dissolve all townships into counties using COUNTYNAME
# Verify: pick Hualien County (花蓮縣) and check its geometry type and area

# Dissolve townships by county name
counties = townships.dissolve(by='COUNTYNAME')

# Filter for Hualien County (花蓮縣)
hualien = counties[counties.index == '花蓮縣']

if len(hualien) > 0:
    hualien_geom = hualien.geometry.iloc[0]
    hualien_area_km2 = hualien_geom.area / 1_000_000

    print(f"Hualien County (花蓮縣):")
    print(f"Geometry type: {hualien_geom.geom_type}")
    print(f"Area: {hualien_area_km2:.2f} km²")
else:
    print("Hualien County not found in the data")

### Step 2: Create synthetic shelter data for Hualien

In [ ]:
# Lab 1 Step 2: YOUR CODE HERE
# Create synthetic shelter points in Hualien County
# Each town should have 2-8 shelters with random capacity (50-500)
# Place shelters at each town's centroid (simplified)

import numpy as np
from shapely.geometry import Point

# Set random seed for reproducibility
np.random.seed(42)

# Filter townships for Hualien County
hualien_towns = townships[townships['COUNTYNAME'] == '花蓮縣'].copy()

shelters_data = []
shelter_id = 1

# Create shelters for each town in Hualien
for idx, town in hualien_towns.iterrows():
    town_name = town['TOWNNAME']
    town_centroid = town.geometry.centroid

    # Random number of shelters between 2-8
    num_shelters = np.random.randint(2, 9)

    # Create shelters at town centroid (simplified approach)
    for i in range(num_shelters):
        # Random capacity between 50-500
        capacity = np.random.randint(50, 501)

        # Add small random offset to avoid exact duplicate points
        offset_x = np.random.uniform(-100, 100)  # meters
        offset_y = np.random.uniform(-100, 100)  # meters
        shelter_point = Point(town_centroid.x + offset_x, town_centroid.y + offset_y)

        shelters_data.append({
            'shelter_id': shelter_id,
            'TOWNNAME': town_name,
            'capacity': capacity,
            'geometry': shelter_point
        })
        shelter_id += 1

# Create GeoDataFrame for shelters
shelters = gpd.GeoDataFrame(shelters_data, crs='EPSG:3826')

print(f"Created {len(shelters)} shelters in {len(hualien_towns)} towns in Hualien County")
print(f"Total capacity: {shelters['capacity'].sum()}")
print("\nFirst 5 shelters:")
print(shelters.head())

### Step 3: Groupby — Statistics per town

In [ ]:
# Lab 1 Step 3: YOUR CODE HERE
# Group shelters by TOWNNAME
# Calculate: count, total capacity, average capacity
# Sort by total capacity descending

# Group shelters by town name
town_summary = shelters.groupby('TOWNNAME').agg(
    shelter_count=('capacity', 'count'),
    total_capacity=('capacity', 'sum'),
    avg_capacity=('capacity', 'mean')
).round(2)

# Sort by total capacity descending
town_summary = town_summary.sort_values('total_capacity', ascending=False)

print("Shelter statistics by town (sorted by total capacity):")
print(town_summary)
print(f"\nTotal towns with shelters: {len(town_summary)}")
print(f"Total shelters in Hualien: {town_summary['shelter_count'].sum()}")
print(f"Total capacity in Hualien: {town_summary['total_capacity'].sum()}")

### Step 4: Merge stats back to geometry & visualize

In [ ]:
# Lab 1 Step 4: YOUR CODE HERE
# 1. Merge the groupby statistics back to the Hualien township geometry
# 2. Create an interactive map with .explore() showing total_capacity by color
# 3. Save the map as HTML (for Windsurf/VS Code users)

import os

# Create outputs directory if it doesn't exist
os.makedirs('outputs', exist_ok=True)

# Filter Hualien townships
hualien_towns = townships[townships['COUNTYNAME'] == '花蓮縣'].copy()

# Merge town_summary back to Hualien townships using TOWNNAME
hualien_towns = hualien_towns.merge(town_summary, left_on='TOWNNAME', right_index=True, how='left')

# Fill NaN values (towns with no shelters) with 0
hualien_towns[['shelter_count', 'total_capacity', 'avg_capacity']] = hualien_towns[['shelter_count', 'total_capacity', 'avg_capacity']].fillna(0)

# Convert to EPSG:4326 for .explore()
hualien_towns_4326 = hualien_towns.to_crs('EPSG:4326')

# Create interactive map
m = hualien_towns_4326.explore(
    column='total_capacity',
    cmap='YlOrRd',
    tooltip=['TOWNNAME', 'shelter_count', 'total_capacity', 'avg_capacity'],
    popup=True,
    legend=True,
    style_kwds={'weight': 2, 'color': 'black'},
    highlight_kwds={'fillColor': 'blue', 'fillOpacity': 0.8}
)

# Add shelter points
shelters_4326 = shelters.to_crs('EPSG:4326')
shelters_4326.explore(
    m=m,
    color='red',
    marker_kwds={'radius': 3, 'fill': True, 'fillOpacity': 0.8},
    tooltip=['shelter_id', 'TOWNNAME', 'capacity']
)

# Save the map
m.save('outputs/lab1_hualien_capacity.html')

print("Interactive map created and saved!")
print(f"Map shows {len(hualien_towns)} towns and {len(shelters)} shelters")
print(f"Towns with shelters: {(hualien_towns['shelter_count'] > 0).sum()}")
print(f"Towns without shelters: {(hualien_towns['shelter_count'] == 0).sum()}")

# Display the map
display(m)

---

## Cell [3] — The Raster Glass Box: It's Just a Matrix

A raster (like a DEM) is a **NumPy array with a GPS**. Each pixel = one numeric value.

In [ ]:
# Cell [3]: YOUR CODE HERE
# Create a synthetic 100x80 DEM with rioxarray
# Use valley profile: low center, mountains on sides
# Set CRS to EPSG:3826
# Print: shape, CRS, resolution, min/max elevation, top-left 5x5 values

import numpy as np
import xarray as xr
import rioxarray

# Create meshgrids
ny, nx = 80, 100
y = np.linspace(0, 1, ny)
x = np.linspace(0, 1, nx)
X, Y = np.meshgrid(x, y)

# Valley elevation formula
elevation = 50 + 800*(2*X-1)**2 + 200*Y + 30*np.sin(8*X)*np.cos(6*Y)

# Define bounds
x_bounds = [302000, 303600]  # 1.6km width
y_bounds = [2638000, 2640000]  # 2km height

# Calculate resolution
x_res = (x_bounds[1] - x_bounds[0]) / nx
y_res = (y_bounds[1] - y_bounds[0]) / ny

# Create xarray DataArray
dem = xr.DataArray(
    elevation[np.newaxis, :, :],  # Add time dimension
    dims=['band', 'y', 'x'],
    coords={
        'band': [1],
        'x': np.linspace(x_bounds[0], x_bounds[1], nx),
        'y': np.linspace(y_bounds[1], y_bounds[0], ny)  # y coordinates decrease
    }
)

# Set CRS and transform
dem.rio.write_crs('EPSG:3826', inplace=True)
dem.rio.write_transform(inplace=True)

# Print information
print(f"Shape: {dem.shape}")
print(f"CRS: {dem.rio.crs}")
print(f"Resolution: {x_res:.1f}m x {y_res:.1f}m")
print(f"Elevation range: {dem.min().values:.1f}m to {dem.max().values:.1f}m")
print(f"Top-left 5x5 pixel values:")
print(dem.values[0, :5, :5].round(1))

## Cell [4] — Affine Transform

How does `Matrix[row, col]` become real-world `(X, Y)` coordinates?

In [ ]:
# Cell [4]: YOUR CODE HERE
# Read the affine transform from your DEM
# Print the 6 parameters and explain what each means
# Manually convert pixel [5, 10] to real-world coordinates

# Get the affine transform
transform = dem.rio.transform()

# Extract the 6 parameters
a, b, c, d, e, f = transform[:6]

print("Affine Transform Parameters:")
print(f"a (pixel width): {a:.1f} meters")
print(f"b (rotation X): {b:.1f}")
print(f"c (top-left X): {c:.1f}")
print(f"d (rotation Y): {d:.1f}")
print(f"e (pixel height): {e:.1f} meters")
print(f"f (top-left Y): {f:.1f}")

print("\nExplanation:")
print("- a: pixel width in X direction (meters)")
print("- b: rotation/skew in X direction (usually 0)")
print("- c: X coordinate of top-left pixel corner")
print("- d: rotation/skew in Y direction (usually 0)")
print("- e: pixel height in Y direction (negative for north-up)")
print("- f: Y coordinate of top-left pixel corner")

# Manual conversion for pixel [5, 10] (row=5, col=10)
row, col = 5, 10
real_x = a*col + b*row + c
real_y = d*col + e*row + f

print(f"\nManual conversion for pixel [{row}, {col}]:")
print(f"Real-world X: {real_x:.1f}")
print(f"Real-world Y: {real_y:.1f}")

# Get elevation at that pixel
elevation_value = dem.values[0, row, col]
print(f"Elevation at pixel [{row}, {col}]: {elevation_value:.1f} meters")

# Verify with rioxarray
x_coords = dem.rio.transform() * (col, row)
print(f"\nVerification with rioxarray:")
print(f"X coordinate: {x_coords[0]:.1f}")
print(f"Y coordinate: {x_coords[1]:.1f}")

## Cell [5] — DEM Visualization

Create a side-by-side plot: elevation + hillshade.

In [ ]:
# Cell [5]: YOUR CODE HERE
# Create a 1x2 subplot: left = elevation with 'terrain' colormap, right = hillshade
# Save as outputs/cell5_dem.png

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import os

# Create outputs directory if it doesn't exist
os.makedirs('outputs', exist_ok=True)

# Create figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Elevation map
elevation_data = dem.values[0]  # Remove band dimension
im1 = ax1.imshow(elevation_data, cmap='terrain', origin='upper')
ax1.set_title('DEM Elevation')
ax1.set_xlabel('Pixel Column')
ax1.set_ylabel('Pixel Row')

# Add colorbar for elevation
cbar1 = plt.colorbar(im1, ax=ax1, label='Elevation (m)')

# Right: Hillshade
ls = LightSource(azdeg=315, altdeg=45)
hillshade = ls.hillshade(elevation_data, vert_exag=10, dx=16, dy=25, fraction=1.0)

im2 = ax2.imshow(hillshade, cmap='gray', origin='upper')
ax2.set_title('Hillshade')
ax2.set_xlabel('Pixel Column')
ax2.set_ylabel('Pixel Row')

# Add colorbar for hillshade
cbar2 = plt.colorbar(im2, ax=ax2, label='Illumination')

plt.tight_layout()

# Save the figure
plt.savefig('outputs/cell5_dem.png', dpi=150, bbox_inches='tight')
print("DEM visualization saved as outputs/cell5_dem.png")

# Display the plot
plt.show()

---

## 🔬 Lab 2: Raster Exploration on Colab (15 minutes)

### ⚡ 請選擇你的環境：

| | **Option A: Google Colab（建議）** | **Option B: 本機 fallback** |
|---|---|---|
| 適用情境 | 網路正常、Drive mount 成功 | Colab 有問題或無法連線 |
| DEM 來源 | `dem_20m_hualien.tif`（Pre-lab 已下載到 Drive） | 使用 Cell [3] 的合成 DEM (`dem_synth`) |
| 後續 Step 2-4 | 用 `dem` 變數 | 用 `dem_synth` 變數（已在前面建立） |

> **Fallback 快速切換**：如果 Colab 不通，跳過 Step 1，直接到 Step 2，把 `dem` 改成 `dem_synth` 即可繼續。
>
> **Note**: Steps 5-6 (zonal stats & risk classification) are included for reference — they will be completed in the homework.

### Step 1: Colab Setup

In [ ]:
# Lab 2 Step 1: Run in Colab!
# 1. Install packages: !pip install rioxarray rasterstats geopandas -q
# 2. Mount Google Drive
# 3. Import all packages

# AI Prompt suggestion:
# "Set up my Colab environment:
#  1. pip install rioxarray rasterstats geopandas
#  2. Mount Google Drive with drive.mount('/content/drive')
#  3. Import rioxarray, geopandas, numpy, matplotlib.pyplot, rasterstats"

# 1. 安裝套件
!pip install -q rioxarray rasterstats geopandas

# 2. 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. 匯入套件
import rioxarray as rxr
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from rasterstats import zonal_stats

### Step 2: Load DEM & Inspect

In [ ]:
# Lab 2 Step 2: YOUR CODE HERE
# Load the pre-clipped Hualien DEM from Google Drive
# Print: shape, CRS, transform, bounds, elevation range, total pixels

# === Option A: Colab ===
# dem = rioxarray.open_rasterio('/content/drive/MyDrive/GIS_data/dem_20m_hualien.tif')

# === Option B: Local fallback ===
# dem = dem_synth  # 使用 Cell [3] 建立的合成 DEM

# AI Prompt suggestion:
# "Load a GeoTIFF DEM with rioxarray.open_rasterio().
#  Print its shape, CRS, transform, bounds, min/max elevation, and total pixel count."

import rioxarray as rxr
import numpy as np

# 讀取 GeoTIFF DEM
dem = rxr.open_rasterio("/content/drive/MyDrive/GIS_data/dem_20m_hualien.tif")

# 如果 DEM 只有一個 band，通常會保留 band 維度
print("Shape:", dem.shape)
print("CRS:", dem.rio.crs)
print("Transform:", dem.rio.transform())
print("Bounds:", dem.rio.bounds())

# 取得高程最小最大值
print("Min elevation:", float(np.nanmin(dem.values)))
print("Max elevation:", float(np.nanmax(dem.values)))

# 總 pixel 數
total_pixels = dem.shape[-2] * dem.shape[-1]
print("Total pixel count:", total_pixels)


### Step 3: Visualize DEM

In [ ]:
# Lab 2 Step 3: YOUR CODE HERE
# Create a DEM elevation map with colorbar

# AI Prompt suggestion:
# "Visualize my DEM with matplotlib imshow using 'terrain' colormap.
#  Add colorbar labeled 'Elevation (m)'. Figure size 10x8."

import matplotlib.pyplot as plt

# 如果 DEM 是單一 band，先去掉 band 維度
dem_plot = dem.squeeze()

plt.figure(figsize=(10, 8))
img = plt.imshow(dem_plot, cmap='terrain')
plt.colorbar(img, label='Elevation (m)')
plt.title('DEM Visualization')
plt.xlabel('Column')
plt.ylabel('Row')
plt.show()


### Step 4: Compute Slope

In [ ]:
# Lab 2 Step 4: YOUR CODE HERE
# Compute slope from DEM using np.gradient
# IMPORTANT: pixel spacing = 20 (for 20m DEM)
# Print: slope min, max, mean, % pixels > 30°
# Visualize: side-by-side elevation vs slope

# AI Prompt suggestion:
# "Compute slope from my DEM:
#  1. Extract 2D elevation array: dem.values[0]
#  2. Use np.gradient(elevation, 20) — 20 is the pixel spacing in meters
#  3. slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
#  4. Print min, max, mean slope and percentage of pixels > 30 degrees
#  5. Create 1x2 subplot: elevation (terrain cmap) vs slope (YlOrRd cmap)"

import numpy as np
import matplotlib.pyplot as plt

# 1. 取出 2D 高程矩陣
elevation = dem.values[0]

# 2. 計算 x、y 方向梯度
dy, dx = np.gradient(elevation, 20)

# 3. 計算坡度（度）
slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

# 4. 印出統計值
print("Slope statistics")
print("----------------")
print(f"Min slope: {np.nanmin(slope_deg):.2f} degrees")
print(f"Max slope: {np.nanmax(slope_deg):.2f} degrees")
print(f"Mean slope: {np.nanmean(slope_deg):.2f} degrees")

steep_pct = np.sum(slope_deg > 30) / slope_deg.size * 100
print(f"Percentage of pixels > 30 degrees: {steep_pct:.2f}%")

# 5. 並排畫圖
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

img1 = axes[0].imshow(elevation, cmap='terrain')
axes[0].set_title('Elevation')
axes[0].set_xlabel('Column')
axes[0].set_ylabel('Row')
plt.colorbar(img1, ax=axes[0], label='Elevation (m)')

img2 = axes[1].imshow(slope_deg, cmap='YlOrRd')
axes[1].set_title('Slope (degrees)')
axes[1].set_xlabel('Column')
axes[1].set_ylabel('Row')
plt.colorbar(img2, ax=axes[1], label='Slope (°)')

plt.tight_layout()
plt.show()


### Step 5: Zonal Statistics

In [ ]:
# Lab 2 Step 5: YOUR CODE HERE
# 1. Create shelter points within DEM extent (use synthetic data from earlier)
# 2. Create 200m buffers around shelters
# 3. CHECK CRS ALIGNMENT! (shelter CRS must match DEM CRS)
# 4. Run zonal_stats for elevation: mean, min, max, std
# 5. Run zonal_stats for slope: mean, max
# 6. Merge results back to shelter GeoDataFrame

# AI Prompt suggestion:
# "I have shelter points (EPSG:3826) and a DEM GeoTIFF.
#  1. First verify both are in the same CRS
#  2. Create 200m buffers around shelters
#  3. Save DEM and slope as temporary GeoTIFFs
#  4. Use rasterstats.zonal_stats() to get mean/min/max/std elevation per buffer
#  5. Use zonal_stats() to get mean/max slope per buffer
#  6. Merge results back and print a table with shelter_id, mean_elev, max_slope"


### Step 6: Risk Classification & Visualization

In [ ]:
# Lab 2 Step 6: YOUR CODE HERE
# 1. Classify terrain risk: HIGH (slope>30°), MEDIUM (20-30°), LOW (<20°)
# 2. Print risk summary
# 3. Create scatter plot: mean_elevation vs max_slope, colored by risk level
# 4. Save as outputs/lab2_terrain_risk.png

# AI Prompt suggestion:
# "Classify my shelters by terrain risk:
#  - HIGH: max_slope > 30°
#  - MEDIUM: max_slope 20-30°
#  - LOW: max_slope < 20°
#  Print the risk distribution.
#  Create a scatter plot of mean_elev vs max_slope, colored by risk (red/orange/green).
#  Add a horizontal line at slope=30° threshold.
#  Save as outputs/lab2_terrain_risk.png"


---

## Week 4 Reflection

Answer in the markdown cell below:

### My Reflection (fill in)

1. **Why is "distance to river" alone insufficient for flood risk assessment?**
   
   *Your answer:*

2. **What causes zonal_stats to return NaN, and how do you fix it?**
   
   *Your answer:*

3. **When combining vector and raster, should you reproject the vector or the raster? Why?**
   
   *Your answer:*

4. **What was the most surprising thing you learned about raster data today?**
   
   *Your answer:*